In [29]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os



In [ ]:
master_df = pd.read_csv("medicines_master.csv")

print("Total products:", len(master_df))
print("Unique NPL IDs:", master_df["npl_id"].nunique())

master_df.head()


Total products: 15190
Unique NPL IDs: 15190


,product_name,strength,form,atc_code,active_ingredient_simple,active_ingredient,registration_status,sales_status,human_vet,prescription_status,npl_id
0,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20150416000034
1,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20180129000014
2,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20171222000018
3,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20141230000119
4,Abacavir/Lamivudine STADA,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20151021000044


In [ ]:
master_df["npl_id"] = (
    master_df["npl_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

master_df[["product_name", "strength", "form", "npl_id"]].head()


,product_name,strength,form,npl_id
0,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,20150416000034
1,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,20180129000014
2,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,20171222000018
3,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,20141230000119
4,Abacavir/Lamivudine STADA,600 mg/300 mg,Filmdragerad tablett,20151021000044


In [ ]:
print(master_df["npl_id"].str.len().value_counts().sort_index())


npl_id
14    15190
Name: count, dtype: int64


In [ ]:
def get_fass_text(npl_id):
    url = f"https://fass.se/health/product/{npl_id}/smpc"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    text = soup.get_text(
        " ",
        strip=True
    )

    return text


In [ ]:
test_row = master_df.iloc[0]

print("Medicine:", test_row["product_name"])
print("NPL ID:", test_row["npl_id"])

test_text = get_fass_text(test_row["npl_id"])

print("Characters downloaded:", len(test_text))
print(test_text[:1000])


Medicine: Abacavir/Lamivudine Accord
NPL ID: 20150416000034
Characters downloaded: 144582
Abacavir/Lamivudine Accord (Filmdragerad tablett 600 mg/300 mg) • Produktresumé | FASS Vård Tillgänglig läkemedelsinformation i Sverige Kontakta oss FAQ Fass Vård Fass Patient Vård Djur MENY / Läkemedel / Abacavir/Lamivudine Accord, Filmdragerad tablett 600 mg/300 mg / Produktresumé Bild saknas orange, kapselformad, 19,4 mm x 10,4 mm orange, kapselformad, 19,4 mm x 10,4 mm Info om att dela och krossa Accord Healthcare AB Abacavir/Lamivudine Accord, Filmdragerad tablett 600 mg/300 mg Antiviralt medel Bild saknas orange, kapselformad, 19,4 mm x 10,4 mm orange, kapselformad, 19,4 mm x 10,4 mm Info om att dela och krossa Om läkemedlet Recept krävs Ingår i högkostnadsskyddet Läkemedel från Accord Healthcare AB omfattas av Läkemedelsförsäkringen Visa alla Beskrivning Antiviralt medel ATC-kod J05AR02 Utbytbara läkemedel Visa 7 st Substanser i alfabetisk ordning Abakavir Lamivudin Abakavir +1 Abakavir Lam

In [ ]:
def extract_section(text, start_heading, end_heading):
    start = text.find(start_heading)

    if start == -1:
        return None

    end = text.find(end_heading, start + len(start_heading))

    if end == -1:
        return None

    return text[start:end].strip()


In [ ]:
sections = {
    "dosage": (
        "4.2 Dosering och administreringssätt",
        "4.3 Kontraindikationer"
    ),

    "contraindications": (
        "4.3 Kontraindikationer",
        "4.4 Varningar och försiktighet"
    ),

    "warnings": (
        "4.4 Varningar och försiktighet",
        "4.5 Interaktioner med andra läkemedel och övriga interaktioner"
    ),

    "interactions": (
        "4.5 Interaktioner med andra läkemedel och övriga interaktioner",
        "4.6 Fertilitet, graviditet och amning"
    ),

    "pregnancy_breastfeeding": (
        "4.6 Fertilitet, graviditet och amning",
        "4.7 Effekter på förmågan att framföra fordon och använda maskiner"
    ),

    "side_effects": (
        "4.8 Biverkningar",
        "4.9 Överdosering"
    )
}


In [ ]:
test_sections = {}

for section_name, (start, end) in sections.items():

    extracted = extract_section(
        test_text,
        start,
        end
    )

    test_sections[section_name] = extracted

    if extracted:
        print(
            section_name,
            "-> OK | characters:",
            len(extracted)
        )
    else:
        print(section_name, "-> NOT FOUND")


dosage -> OK | characters: 4624
contraindications -> OK | characters: 138
warnings -> OK | characters: 12046
interactions -> OK | characters: 7129
pregnancy_breastfeeding -> OK | characters: 2759
side_effects -> OK | characters: 6941


In [ ]:
print(test_sections["interactions"][:2000])


4.5 Interaktioner med andra läkemedel och övriga interaktioner Abacavir/Lamivudine Accord innehåller abakavir och lamivudin och därför är alla tidigare observerade interaktioner för respektive substans relevanta även för Abacavir/Lamivudine Accord. Kliniska studier har visat att det inte finns några kliniskt relevanta interaktioner mellan abakavir och lamivudin. Abakavir metaboliseras via UDP‑glukuronyltransferas (UGT)‑enzymer och alkoholdehydrogenas; samtidig administrering med inducerare eller hämmare av UGT‑enzymer eller med substanser som elimineras via alkoholdehydrogenas kan påverka exponeringen av abakavir. Lamivudin elimineras renalt. Aktiv renal sekretion av lamivudin i urinen medieras genom organiska katjontransportörer (OCT); samtidig administrering av lamivudin och hämmare av OCT kan öka exponeringen av lamivudin. Abakavir och lamivudin metaboliseras inte till någon betydande grad via P450‑enzymer (såsom CYP 3A4, CYP 2C9 eller CYP 2D6). De inducerar inte heller detta enzyms

50-product pilot test করি।

In [ ]:
pilot_df = master_df.head(50).copy()

pilot_results = []
pilot_errors = []

for i, row in pilot_df.iterrows():

    name = row["product_name"]
    npl_id = row["npl_id"]

    try:
        text = get_fass_text(npl_id)

        result = {
            "product_name": name,
            "strength": row["strength"],
            "form": row["form"],
            "atc_code": row["atc_code"],
            "active_ingredient": row["active_ingredient"],
            "prescription_status": row["prescription_status"],
            "npl_id": npl_id,
            "source_url": f"https://fass.se/health/product/{npl_id}/smpc"
        }

        # Extract all FASS sections
        for section_name, (start, end) in sections.items():
            result[section_name] = extract_section(
                text,
                start,
                end
            )

        pilot_results.append(result)

        print(name, "-> OK")

    except Exception as e:
        pilot_errors.append({
            "product_name": name,
            "npl_id": npl_id,
            "error": str(e)
        })

        print(name, "-> ERROR:", e)

    # Avoid sending requests too quickly
    time.sleep(1)


Abacavir/Lamivudine Accord -> OK
Abacavir/Lamivudine Glenmark -> OK
Abacavir/Lamivudine Medical Valley -> OK
Abacavir/Lamivudine Sandoz -> OK
Abacavir/Lamivudine STADA -> OK
Abakavir/Lamivudin Viatris -> OK
ABASAGLAR -> OK
ABASAGLAR KwikPen -> OK
ABASAGLAR Tempo Pen -> OK
Abavalex -> OK
Abavalex -> OK
Abavalex -> OK
Abavalex -> OK
Abavalex -> OK
Abbonate -> OK
Abboticin -> OK
Abboticin -> OK
Abboticin Novum -> OK
Abdomin Citrus -> OK
Abdomin Junior Neutral -> OK
Abdomin Neutral -> OK
Abecma -> OK
Abecol -> OK
Abetip -> OK
Abetip -> OK
Abevmy -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
Abilify -> OK
ABILIFY MAINTENA -> OK
ABILIFY MAINTENA -> OK
ABILIFY MAINTENA -> OK
ABILIFY MAINTENA -> OK
ABILIFY MAINTENA -> OK
ABILIFY MAINTENA -> OK
Abirateron Avansor -> OK
Abirateron Devatis -> OK
Abirateron Medical Valley -> OK
Abirateron Sandoz -> OK
Abirateron Teva -> OK
Abiraterone Accord -> OK
Abiraterone Accord -> OK
Abir

In [ ]:
pilot_result_df = pd.DataFrame(pilot_results)

print("Attempted:", len(pilot_df))
print("Downloaded:", len(pilot_result_df))
print("Request errors:", len(pilot_errors))

pilot_result_df.head()


Attempted: 50
Downloaded: 50
Request errors: 0


,product_name,strength,form,atc_code,active_ingredient,prescription_status,npl_id,source_url,dosage,contraindications,warnings,interactions,pregnancy_breastfeeding,side_effects
0,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,abakavir\nlamivudin,Receptbelagt,20150416000034,https://fass.se/health/product/20150416000034/...,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
1,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,abakavirhydrokloridmonohydrat\nlamivudin,Receptbelagt,20180129000014,https://fass.se/health/product/20180129000014/...,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
2,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,abakavirsulfat\nlamivudin,Receptbelagt,20171222000018,https://fass.se/health/product/20171222000018/...,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
3,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,abakavirhydrokloridmonohydrat\nlamivudin,Receptbelagt,20141230000119,https://fass.se/health/product/20141230000119/...,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
4,Abacavir/Lamivudine STADA,600 mg/300 mg,Filmdragerad tablett,J05AR02,abakavir\nlamivudin,Receptbelagt,20151021000044,https://fass.se/health/product/20151021000044/...,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...


In [ ]:
section_columns = [
    "dosage",
    "contraindications",
    "warnings",
    "interactions",
    "pregnancy_breastfeeding",
    "side_effects"
]

print("\nMissing sections:")

for col in section_columns:
    missing = pilot_result_df[col].isna().sum()
    print(f"{col}: {missing}")



Missing sections:
dosage: 12
contraindications: 12
warnings: 12
interactions: 12
pregnancy_breastfeeding: 12
side_effects: 12


In [ ]:
complete = (
    pilot_result_df[section_columns]
    .notna()
    .all(axis=1)
)

print("\nComplete products:", complete.sum())
print("Incomplete products:", (~complete).sum())



Complete products: 38
Incomplete products: 12


In [ ]:
pilot_result_df.loc[
    ~complete,
    ["product_name", "strength", "form", "npl_id"] + section_columns
]


,product_name,strength,form,npl_id,dosage,contraindications,warnings,interactions,pregnancy_breastfeeding,side_effects
9,Abavalex,10 mg,Filmdragerad tablett,20240402000039,NaN,NaN,NaN,NaN,NaN,NaN
10,Abavalex,100 mg,Filmdragerad tablett,20240402000077,NaN,NaN,NaN,NaN,NaN,NaN
11,Abavalex,25 mg,Filmdragerad tablett,20240402000046,NaN,NaN,NaN,NaN,NaN,NaN
12,Abavalex,50 mg,Filmdragerad tablett,20240402000053,NaN,NaN,NaN,NaN,NaN,NaN
13,Abavalex,75 mg,Filmdragerad tablett,20240402000060,NaN,NaN,NaN,NaN,NaN,NaN
18,Abdomin Citrus,NaN,Pulver till oral lösning i dospåse,20240318000178,NaN,NaN,NaN,NaN,NaN,NaN
23,Abetip,10 mg,Filmdragerad tablett,20210429000176,NaN,NaN,NaN,NaN,NaN,NaN
24,Abetip,20 mg,Filmdragerad tablett,20210429000183,NaN,NaN,NaN,NaN,NaN,NaN
42,Abirateron Devatis,500 mg,Filmdragerad tablett,20250508000031,NaN,NaN,NaN,NaN,NaN,NaN
45,Abirateron Teva,500 mg,Filmdragerad tablett,20201201000118,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
failed_npl = "20240402000039"

failed_text = get_fass_text(failed_npl)

print("Characters:", len(failed_text))
print(failed_text[:2000])


Characters: 1689
Abavalex (Filmdragerad tablett 10 mg) • Produktresumé | FASS Vård Tillgänglig läkemedelsinformation i Sverige Kontakta oss FAQ Fass Vård Fass Patient Vård Djur MENY / Läkemedel / Abavalex, Filmdragerad tablett 10 mg / Produktresumé Bild saknas Info om att dela och krossa G.L. Pharma Nordic Abavalex, Filmdragerad tablett 10 mg Tillhandahålls ej Bild saknas Info om att dela och krossa Om läkemedlet Läkemedel från G.L. Pharma Nordic omfattas av Läkemedelsförsäkringen Visa alla ATC-kod N03AX23 Utbytbara läkemedel Ej utbytbar Substans Brivaracetam Laddar... Fass Tillgänglig läkemedelsinformation i Sverige. Fass är den huvudsakliga källan för läkemedelsinformation i Sverige sedan 1966. Fass är en del av Lif – branschorganisationen för de forskande läkemedelsföretagen i Sverige. LinkedIn Följ Fass på Linkedin Versionsnummer 20260820.0856-bde47b434 Kontakta oss Telefon: 08-462 37 00 E-post: info@fass.se Box 17608, 118 92 Stockholm Fler kontaktuppgifter Om webbplatsen Om fass.s

In [ ]:
for heading in [
    "4.1",
    "4.2",
    "4.3",
    "4.4",
    "4.5",
    "4.6",
    "4.7",
    "4.8",
    "4.9"
]:
    print(heading, failed_text.find(heading))


4.1 -1
4.2 -1
4.3 -1
4.4 -1
4.5 -1
4.6 -1
4.7 -1
4.8 -1
4.9 -1


In [ ]:
print("Characters:", len(failed_text))
print("\nFIRST 3000 CHARACTERS:\n")
print(failed_text[:3000])


Characters: 1689

FIRST 3000 CHARACTERS:

Abavalex (Filmdragerad tablett 10 mg) • Produktresumé | FASS Vård Tillgänglig läkemedelsinformation i Sverige Kontakta oss FAQ Fass Vård Fass Patient Vård Djur MENY / Läkemedel / Abavalex, Filmdragerad tablett 10 mg / Produktresumé Bild saknas Info om att dela och krossa G.L. Pharma Nordic Abavalex, Filmdragerad tablett 10 mg Tillhandahålls ej Bild saknas Info om att dela och krossa Om läkemedlet Läkemedel från G.L. Pharma Nordic omfattas av Läkemedelsförsäkringen Visa alla ATC-kod N03AX23 Utbytbara läkemedel Ej utbytbar Substans Brivaracetam Laddar... Fass Tillgänglig läkemedelsinformation i Sverige. Fass är den huvudsakliga källan för läkemedelsinformation i Sverige sedan 1966. Fass är en del av Lif – branschorganisationen för de forskande läkemedelsföretagen i Sverige. LinkedIn Följ Fass på Linkedin Versionsnummer 20260820.0856-bde47b434 Kontakta oss Telefon: 08-462 37 00 E-post: info@fass.se Box 17608, 118 92 Stockholm Fler kontaktuppgifter

In [ ]:
keywords = [
    "Produktresumé",
    "Fass-text",
    "Abavalex",
    "Laddar",
    "Dosering",
    "Kontraindikationer",
    "Interaktioner"
]

for word in keywords:
    print(word, "->", failed_text.find(word))


Produktresumé -> 40
Fass-text -> 1238
Abavalex -> 0
Laddar -> 549
Dosering -> -1
Kontraindikationer -> -1
Interaktioner -> -1


In [ ]:
def get_fass_status(text):
    if "Fass har begränsad information om detta läkemedel" in text:
        return "limited_information"

    if "4.2 Dosering och administreringssätt" in text:
        return "smpc_available"

    return "unknown"


In [ ]:
print("Abacavir:", get_fass_status(test_text))
print("Abavalex:", get_fass_status(failed_text))


Abacavir: smpc_available
Abavalex: limited_information


In [ ]:
incomplete_df = pilot_result_df.loc[
    ~complete,
    ["product_name", "npl_id"]
].copy()

status_results = []

for _, row in incomplete_df.iterrows():
    try:
        text = get_fass_text(row["npl_id"])

        status_results.append({
            "product_name": row["product_name"],
            "npl_id": row["npl_id"],
            "fass_status": get_fass_status(text)
        })

        time.sleep(1)

    except Exception as e:
        status_results.append({
            "product_name": row["product_name"],
            "npl_id": row["npl_id"],
            "fass_status": "request_error"
        })

status_df = pd.DataFrame(status_results)

status_df


,product_name,npl_id,fass_status
0,Abavalex,20240402000039,limited_information
1,Abavalex,20240402000077,limited_information
2,Abavalex,20240402000046,limited_information
3,Abavalex,20240402000053,limited_information
4,Abavalex,20240402000060,limited_information
5,Abdomin Citrus,20240318000178,limited_information
6,Abetip,20210429000176,limited_information
7,Abetip,20210429000183,limited_information
8,Abirateron Devatis,20250508000031,limited_information
9,Abirateron Teva,20201201000118,limited_information


In [ ]:
status_df["fass_status"].value_counts()


fass_status
limited_information    12
Name: count, dtype: int64

In [ ]:


OUTPUT_FILE = "fass_collection.csv"
ERROR_FILE = "fass_errors.csv"

CHECKPOINT_EVERY = 50
REQUEST_DELAY = 1


In [ ]:
def collect_product(row):

    npl_id = str(row["npl_id"])
    url = f"https://fass.se/health/product/{npl_id}/smpc"

    result = {
        "product_name": row["product_name"],
        "strength": row["strength"],
        "form": row["form"],
        "atc_code": row["atc_code"],
        "active_ingredient_simple": row["active_ingredient_simple"],
        "active_ingredient": row["active_ingredient"],
        "prescription_status": row["prescription_status"],
        "sales_status": row["sales_status"],
        "npl_id": npl_id,
        "source_url": url,

        "fass_status": None,

        "dosage": None,
        "contraindications": None,
        "warnings": None,
        "interactions": None,
        "pregnancy_breastfeeding": None,
        "side_effects": None
    }

    text = get_fass_text(npl_id)

    status = get_fass_status(text)
    result["fass_status"] = status

    if status == "smpc_available":

        for section_name, (start, end) in sections.items():

            result[section_name] = extract_section(
                text,
                start,
                end
            )

    return result


In [ ]:


if os.path.exists(OUTPUT_FILE):

    existing_df = pd.read_csv(
        OUTPUT_FILE,
        dtype={"npl_id": str}
    )

    completed_ids = set(
        existing_df["npl_id"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )

    results = existing_df.to_dict("records")

    print(
        "Resuming from previous collection:",
        len(completed_ids),
        "products already processed."
    )

else:

    completed_ids = set()
    results = []

    print("Starting new FASS collection.")



errors = []




processed_since_save = 0

for index, row in master_df.iterrows():

    npl_id = str(row["npl_id"])

    # Already collected → skip
    if npl_id in completed_ids:
        continue

    try:

        result = collect_product(row)

        results.append(result)
        completed_ids.add(npl_id)

        processed_since_save += 1

        print(
            f"[{len(completed_ids)}/{len(master_df)}]",
            row["product_name"],
            "->",
            result["fass_status"]
        )

    except Exception as e:

        errors.append({
            "product_name": row["product_name"],
            "npl_id": npl_id,
            "error": str(e)
        })

        print(
            f"[ERROR] {row['product_name']} | {npl_id}",
            str(e)
        )


    if processed_since_save >= CHECKPOINT_EVERY:

        pd.DataFrame(results).to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        if errors:
            pd.DataFrame(errors).to_csv(
                ERROR_FILE,
                index=False,
                encoding="utf-8-sig"
            )

        print(
            "\n--- CHECKPOINT SAVED:",
            len(completed_ids),
            "products ---\n"
        )

        processed_since_save = 0

    time.sleep(REQUEST_DELAY)


Starting new FASS collection.
[1/15190] Abacavir/Lamivudine Accord -> smpc_available
[2/15190] Abacavir/Lamivudine Glenmark -> smpc_available
[3/15190] Abacavir/Lamivudine Medical Valley -> smpc_available
[4/15190] Abacavir/Lamivudine Sandoz -> smpc_available
[5/15190] Abacavir/Lamivudine STADA -> smpc_available
[6/15190] Abakavir/Lamivudin Viatris -> smpc_available
[7/15190] ABASAGLAR -> smpc_available
[8/15190] ABASAGLAR KwikPen -> smpc_available
[9/15190] ABASAGLAR Tempo Pen -> smpc_available
[10/15190] Abavalex -> limited_information
[11/15190] Abavalex -> limited_information
[12/15190] Abavalex -> limited_information
[13/15190] Abavalex -> limited_information
[14/15190] Abavalex -> limited_information
[15/15190] Abbonate -> smpc_available
[16/15190] Abboticin -> smpc_available
[17/15190] Abboticin -> smpc_available
[18/15190] Abboticin Novum -> smpc_available
[19/15190] Abdomin Citrus -> limited_information
[20/15190] Abdomin Junior Neutral -> smpc_available
[21/15190] Abdomin Neu

In [ ]:
fass_all = pd.read_csv(
    "fass_collection.csv",
    dtype={"npl_id": str}
)

print("Collected rows:", len(fass_all))
print(fass_all["fass_status"].value_counts())


Collected rows: 14900
fass_status
smpc_available         9459
limited_information    5150
unknown                 291
Name: count, dtype: int64
